# TRACE — Stage: train (T8 Baselines + T9 Augmentation Ablation)

Run this notebook on a **GPU runtime** (Runtime → Change runtime type → T4 GPU).

Steps:
1. Clone the TRACE repo
2. Upload the dataset zip into Colab
3. Unzip the dataset
4. Run the `train` stage (+ optional `report`)
5. Download `results.zip`

Estimated time: ~2–4 hours for 4 models × 3 seeds × 50 epochs on T4.
Use `--seeds 0` first to verify, then add seeds 1 2.

In [ ]:
# 1. Install dependencies
!pip install -q -U ultralytics

In [ ]:
# 2. Clone the TRACE repo (adjust branch/tag if needed)
!git clone https://github.com/indri007/TRACE.git
%cd TRACE

In [ ]:
# 3. Upload dataset zip from your laptop
from google.colab import files
uploaded = files.upload()   # select vehicle-detection.v1i.yolov12.zip

In [ ]:
# 4. Unzip dataset into ./data
import zipfile, pathlib
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name) as z:
    z.extractall('data')
print(list(pathlib.Path('data').iterdir()))

In [ ]:
# 5a. Quick sanity check — audit + env + eval only (~5 min)
!python trace_experiments.py \
    --data-root ./data \
    --weights app/models/best.pt \
    --out ./results \
    --device 0 \
    --stages audit env eval latency errors report

In [ ]:
# 5b. Training — seed 0 first (verify), then add 1 2
# Estimated: ~30-45 min per model per seed on T4
!python trace_experiments.py \
    --data-root ./data \
    --weights app/models/best.pt \
    --out ./results \
    --device 0 \
    --stages train report \
    --seeds 0

In [ ]:
# 5c. Add seeds 1 and 2 (already-done runs are skipped automatically)
!python trace_experiments.py \
    --data-root ./data \
    --weights app/models/best.pt \
    --out ./results \
    --device 0 \
    --stages train report \
    --seeds 0 1 2

In [ ]:
# 6. Download results
from google.colab import files
files.download('results.zip')

## After downloading results.zip

1. Place `results.zip` in your TRACE repo root.
2. Unzip: `unzip -o results.zip -d results/`
3. The tables `T8_baseline_comparison.md` and `T9_augmentation_ablation.md`
   will be in `results/tables/`.
4. Copy the numbers into the manuscript:
   - `manuscript/sections/04_experiments.tex` — replace [DATA DIBUTUHKAN] in T8/T9 descriptions
   - `manuscript/sections/05_results.tex` — add Table T8 and T9
   - `manuscript/sections/06_discussion.tex` — fill RQ1 baseline comparison
5. Commit: `git add results/tables/ manuscript/ && git commit -m 'Add T8/T9 training results'`